# がんばる日本語ラボをGoogle Colabで起動する

このノートブックは、StreamlitアプリをColab上で起動し、外部から開ける一時URLを発行します。

- `app.py`、`curriculum.py`、`requirements.txt` が必要です。
- GitHubリポジトリがある場合は、次のセルの `REPO_URL` にURLを入れてください。
- GitHubがない場合は、次のセルの実行中にファイルアップロードできます。
- 発行されるURLはColabセッションが動いている間だけ有効です。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

# GitHubに置いた場合は、ここにリポジトリURLを入れます。
# 例: REPO_URL = "https://github.com/your-name/gannbaru-streamlit-app.git"
REPO_URL = ""  # @param {type:"string"}
PROJECT_DIR = Path("/content/gannbaru-streamlit-app")

if REPO_URL.strip():
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(["git", "clone", REPO_URL.strip(), str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)
else:
    print("GitHub URLが空なので、現在のフォルダかアップロード済みファイルを使います。")
    missing = [name for name in ["app.py", "curriculum.py"] if not Path(name).exists()]
    if missing:
        print("不足ファイル:", missing)
        print("app.py、curriculum.py、requirements.txt を選んでアップロードしてください。")
        from google.colab import files
        files.upload()

if not Path("requirements.txt").exists():
    Path("requirements.txt").write_text("streamlit>=1.35\n", encoding="utf-8")

required = ["app.py", "curriculum.py", "requirements.txt"]
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError(f"必要なファイルが見つかりません: {missing}")

print("準備OK:", Path.cwd())
print("ファイル:", ", ".join(required))

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import socket
import subprocess
import sys
import time
from pathlib import Path

PORT = 8501

def wait_for_port(port: int, timeout: int = 30) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(1)
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(1)
    return False

log_file = open("streamlit.log", "w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.address",
        "0.0.0.0",
        "--server.port",
        str(PORT),
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

if not wait_for_port(PORT):
    log_file.flush()
    print(Path("streamlit.log").read_text(encoding="utf-8", errors="replace")[-3000:])
    raise RuntimeError("Streamlitが起動しませんでした。ログを確認してください。")

print(f"Streamlit started on port {PORT}")

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import re
import subprocess
import time

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(120):
    line = tunnel_proc.stdout.readline()
    if line:
        print(line, end="")
        match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if not public_url:
    raise RuntimeError("公開URLを取得できませんでした。セルをもう一度実行してください。")

print("\n公開URL:", public_url)
print("このURLはColabが動いている間だけ使えます。")

## 停止する場合

次のセルを実行すると、Streamlitと公開トンネルを止めます。

In [ ]:
for proc_name in ["tunnel_proc", "streamlit_proc"]:
    proc = globals().get(proc_name)
    if proc and proc.poll() is None:
        proc.terminate()
        print(proc_name, "stopped")